# `ppi_aipw` quickstart

A compact walkthrough of the two main entry points:

- `mean_inference(...)` for semisupervised mean estimation
- `causal_inference(...)` for arm-specific means and control-vs-treatment ATEs

The examples stay small enough to skim, but they use the current API and the same defaults highlighted on the website.

## Package map

Use the mean API when you have:

- `Y`: observed outcomes on labeled rows
- `Yhat`: predictions on those same rows
- `Yhat_unlabeled`: predictions on a larger unlabeled sample

Use the causal API when you have:

- `Y`: observed outcomes
- `A`: observed treatment assignments
- `Yhat_potential`: one prediction column per treatment arm

Practical defaults:

- start with `method="monotone_spline"` for the mean API
- use `method="linear"` when you want the simplest recalibration
- use `method="prognostic_linear"` when you want the score plus extra covariates
- use `inference="wald"` for a fast interval; the mean API also supports `jackknife` and `bootstrap`

If you only need one output at a time, the package also exports `mean_pointestimate`, `mean_se`, and `mean_ci`.

In [ ]:
import numpy as np

from ppi_aipw import calibration_diagnostics, causal_inference, mean_inference, plot_calibration

## Mean API

The main workflow is:

1. collect outcomes `Y` on the labeled sample
2. collect predictions `Yhat` on those same labeled rows
3. collect predictions `Yhat_unlabeled` on the unlabeled rows
4. call `mean_inference(...)`

Here we simulate a setting where the score is helpful but slightly miscalibrated, so the package default `method="monotone_spline"` is a good first choice.

In [ ]:
rng = np.random.default_rng(7)

n_labeled = 250
n_unlabeled = 2000

x_labeled = rng.normal(size=(n_labeled, 2))
x_unlabeled = rng.normal(size=(n_unlabeled, 2))

mu_labeled = 3.0 + 1.1 * x_labeled[:, 0] - 0.7 * x_labeled[:, 1] + 0.3 * x_labeled[:, 0] * x_labeled[:, 1]
mu_unlabeled = 3.0 + 1.1 * x_unlabeled[:, 0] - 0.7 * x_unlabeled[:, 1] + 0.3 * x_unlabeled[:, 0] * x_unlabeled[:, 1]

Y = mu_labeled + rng.normal(scale=0.9, size=n_labeled)

# The raw score is useful, but nonlinear and slightly shifted.
Yhat = 0.85 * mu_labeled + 0.35 * np.maximum(mu_labeled - 3.0, 0.0) + 0.4 + rng.normal(scale=0.45, size=n_labeled)
Yhat_unlabeled = 0.85 * mu_unlabeled + 0.35 * np.maximum(mu_unlabeled - 3.0, 0.0) + 0.4 + rng.normal(scale=0.45, size=n_unlabeled)

In [ ]:
mean_result = mean_inference(
    Y,
    Yhat,
    Yhat_unlabeled,
    method="monotone_spline",
    alpha=0.1,
    inference="wald",
)

mean_summary = {
    "method": mean_result.method,
    "estimate": round(float(mean_result.pointestimate), 3),
    "standard_error": round(float(mean_result.se), 3),
    "90%_ci": tuple(round(float(x), 3) for x in mean_result.ci),
    "wald_p_value_against_0": round(float(mean_result.diagnostics["wald_p_value"]), 4),
    "naive_unlabeled_prediction_mean": round(float(np.mean(Yhat_unlabeled)), 3),
    "true_unlabeled_mean_in_this_simulation": round(float(np.mean(mu_unlabeled)), 3),
}

print(mean_result.summary())
mean_summary

This is also a good place to inspect the fitted calibration map on the labeled sample.

- Use `method="linear"` when you want the simplest affine recalibration and maximum interpretability.
- Use `method="monotone_spline"` when you want a smooth monotone default that is more flexible than linear but less stepwise than isotonic.
- Use `method="auto"` when you want the package to choose among a shortlist such as `("aipw", "linear", "monotone_spline", "isotonic")`.
- `calibration_diagnostics(...)` uses only the labeled sample because that is where outcomes are observed.

In [ ]:
mean_diagnostics = calibration_diagnostics(mean_result, Y, Yhat, num_bins=8)

diagnostic_summary = {
    "num_bins": mean_diagnostics.num_bins,
    "first_bin_count": int(mean_diagnostics.per_output[0].bin_counts[0]),
    "first_bin_mean_raw_score": round(float(mean_diagnostics.per_output[0].bin_mean_raw_score[0]), 3),
    "first_bin_mean_outcome": round(float(mean_diagnostics.per_output[0].bin_mean_outcome[0]), 3),
}

diagnostic_summary

In [ ]:
try:
    ax = plot_calibration(mean_diagnostics)
    ax.figure.tight_layout()
except ImportError as exc:
    print(exc)
    print("Install matplotlib if you want the plotting helper in this notebook.")

## Causal API

The causal wrapper reuses the same semisupervised mean engine arm by arm.

You provide:

- `Y`: observed outcomes
- `A`: observed treatment assignments
- `Yhat_potential`: one prediction column per treatment arm

Below, we use `method="prognostic_linear"` so the arm-specific score can borrow strength from extra covariates `X`.

In [ ]:
n = 1500
X = rng.normal(size=(n, 2))

propensity = 1.0 / (1.0 + np.exp(-0.5 * X[:, 0] + 0.2 * X[:, 1]))
A = np.where(rng.uniform(size=n) < propensity, "treated", "control")

mu0 = 1.8 + 0.7 * X[:, 0] - 0.5 * X[:, 1]
tau = 0.6 + 0.25 * X[:, 0]
mu1 = mu0 + tau
Y = np.where(A == "treated", mu1, mu0) + rng.normal(scale=0.8, size=n)

Yhat_potential = np.column_stack([
    0.9 * mu0 + 0.15 * X[:, 0] + 0.3 + rng.normal(scale=0.35, size=n),
    0.9 * mu1 + 0.15 * X[:, 0] + 0.3 + rng.normal(scale=0.35, size=n),
])

causal_result = causal_inference(
    Y,
    A,
    Yhat_potential,
    X=X,
    treatment_levels=("control", "treated"),
    control_arm="control",
    method="prognostic_linear",
    alpha=0.1,
)

print(causal_result.summary())

{
    "control_mean": round(float(causal_result.arm_means["control"]), 3),
    "treated_mean": round(float(causal_result.arm_means["treated"]), 3),
    "treated_vs_control_ate": round(float(causal_result.ate["treated"]), 3),
    "90%_ate_ci": tuple(round(float(x), 3) for x in causal_result.ate_cis["treated"]),
    "true_sample_ate_in_this_simulation": round(float(np.mean(mu1 - mu0)), 3),
}

## What to change for your own data

- Keep rows aligned across outcomes, predictions, treatments, covariates, and weights.
- For the mean API, pass `w` and `w_unlabeled` if you want weighted inference.
- For the causal API, pass one full-sample `w` vector if you already have trial or observational weights.
- If you want data-adaptive method choice for the mean problem, try `method="auto"` with `candidate_methods=("aipw", "linear", "isotonic")`.
- The causal API currently supports `inference="wald"` only.

This notebook is intentionally compact. For fuller argument-by-argument details, see the package website and vignette.